In [ ]:
# Runs to analyse: $NESYSY_RUNS (set by run.py), else output/full, else output/subset.
# Figures and tables are written to <runs>/paper_results/.
import os
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
RUNS = Path(os.environ.get("NESYSY_RUNS") or next(
    (ROOT / "output" / m for m in ("full", "subset") if (ROOT / "output" / m).exists()),
    ROOT / "output" / "subset",
))
# the paper shows only points with all 25 runs; a subset has fewer runs per point
MIN_RUNS = int(os.environ.get("NESYSY_MIN_RUNS", 25 if "full" in RUNS.parts else 1))
OUT = RUNS / "paper_results"
OUT.mkdir(parents=True, exist_ok=True)
DBS = sorted(RUNS.glob("**/mlflow.db"))


def search_runs(experiment):
    """mlflow.search_runs of an experiment, over all databases below RUNS."""
    frames = []
    for db in DBS:
        mlflow.set_tracking_uri(f"sqlite:///{db}")
        exp = mlflow.get_experiment_by_name(experiment)
        if exp is not None:
            frames.append(mlflow.search_runs(experiment_ids=[exp.experiment_id]))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


print(f"runs: {RUNS} ({len(DBS)} database(s)), at least {MIN_RUNS} run(s) per point")


In [ ]:
def load_runs(name):
    runs = search_runs(name)
    metric_cols = [c for c in runs.columns if c.startswith("metrics.")]
    if metric_cols:
        runs = runs[runs[metric_cols[0]].notna()]
    return runs


def delta_table(runs, x, metric, higher_better):
    """Best run per (x, approach, dataset seed) minus the best Base run of the same (x, seed)."""
    if higher_better:
        best = runs.loc[runs.groupby([x, "approach", "params.seed"])[metric].idxmax()]
    else:
        best = runs.loc[runs.groupby([x, "approach", "params.seed"])[metric].idxmin()]

    bl = best[best["approach"] == "plain_nn"][[x, "params.seed", metric]].rename(columns={metric: "baseline"})
    others = best[best["approach"] != "plain_nn"]

    merged = others.merge(bl, on=[x, "params.seed"])
    merged["delta"] = merged[metric] - merged["baseline"]

    delta_summary = merged.groupby([x, "approach"])["delta"].agg(["mean", "sem", "count"]).reset_index()

    if higher_better:
        delta_summary["mean"] *= 100
        delta_summary["sem"] *= 100
    return best, delta_summary


In [ ]:
# Table 2: CNF, Delta accuracy (%) vs Base and VR (%) of Base and BaseSeq
runs = load_runs("cnf_methods")
if runs.empty:
    print("No runs found for cnf_methods, skipping.")
else:
    runs["approach"] = runs["params.model_type"].str.replace(r'_h\d+_l\d+$', '', regex=True)
    best, delta_summary = delta_table(runs, "params.cvr", "metrics.accuracy", higher_better=True)
    filtered = best[best["approach"].isin(["plain_nn", "sequential_beam"])]
    cvr_summary = filtered.groupby(["params.cvr", "approach"])["metrics.constraint_violation_ratio"].agg(["mean", "sem", "count"]).reset_index()
    cvr_summary["mean"] *= 100
    cvr_summary["sem"] *= 100
    delta_summary.to_csv(OUT / "table2_delta_accuracy.csv", index=False)
    cvr_summary.to_csv(OUT / "table2_vr.csv", index=False)
    display(delta_summary, cvr_summary)


In [ ]:
# Table 3: CNF, cvr = 2, Delta accuracy (%) vs Base per training set size (n_train + n_valid)
runs = load_runs("cnf_train_size")
if runs.empty:
    print("No runs found for cnf_train_size, skipping.")
else:
    runs["approach"] = runs["params.model_type"].str.replace(r'_h\d+_l\d+$', '', regex=True)
    runs["size"] = runs["params.n_train"].astype(int) + runs["params.n_valid"].astype(int)
    _, delta_summary = delta_table(runs, "size", "metrics.accuracy", higher_better=True)
    delta_summary.to_csv(OUT / "table3_delta_accuracy.csv", index=False)
    display(delta_summary)


In [ ]:
# Table 4: bounded intervals, Delta MAE vs Base and MCD of Base
runs = load_runs("bi_methods")
if runs.empty:
    print("No runs found for bi_methods, skipping.")
else:
    runs["approach"] = runs["params.model_type"].str.replace(r'_h\d+_l\d+$', '', regex=True)
    best, delta_summary = delta_table(runs, "params.cvr", "metrics.mean_absolute_error", higher_better=False)
    filtered = best[best["approach"].isin(["plain_nn"])]
    mcd_summary = filtered.groupby(["params.cvr", "approach"])["metrics.mean_consistent_distance"].agg(["mean", "sem", "count"]).reset_index()
    delta_summary.to_csv(OUT / "table4_delta_mae.csv", index=False)
    mcd_summary.to_csv(OUT / "table4_mcd.csv", index=False)
    display(delta_summary, mcd_summary)
